In [13]:
import pandas as pd

Too many categories to list

In [14]:
# Option 2: ISO-8859-1 (Latin-1)
df_kaggle = pd.read_csv(
    "/Users/Rex/vscode/ai_lora_tuning/alldata_1_for_kaggle.csv",
    encoding="ISO-8859-1",
    on_bad_lines="skip",
    engine="python"
)

In [15]:
df_kaggle.head(50)

,Unnamed: 0,0,a
0,0,Thyroid_Cancer,Thyroid surgery in children in a single insti...
1,1,Thyroid_Cancer,""" The adopted strategy was the same as that us..."
2,2,Thyroid_Cancer,coronary arterybypass grafting thrombosis ï¬b...
3,3,Thyroid_Cancer,Solitary plasmacytoma SP of the skull is an u...
4,4,Thyroid_Cancer,This study aimed to investigate serum matrix ...
5,5,Thyroid_Cancer,This study was performed to explore the effec...
6,6,Thyroid_Cancer,This study was performed assess the clinical ...
7,7,Thyroid_Cancer,Journal of International Medical Research  Th...
8,8,Thyroid_Cancer,Gastric cancer GC persists as a worldwide pub...
9,9,Thyroid_Cancer,Scars Burns HealingVolume  reuse guideli...


In [16]:
df_kaggle.shape

(7570, 3)

In [18]:
unique = df_kaggle['0'].value_counts()
print(unique)

0
Thyroid_Cancer    2810
Colon_Cancer      2580
Lung_Cancer       2180
Name: count, dtype: int64


In [24]:
import pandas as pd
import re
import os

def clean_biomed_csv(input_csv_path: str, output_csv_path: str = None):
    """
    Cleans a biomedical text CSV by:
      - Handling non-UTF-8 encodings (cp1252, ISO-8859-1)
      - Stripping non-word symbols except punctuation and spaces
      - Saving the cleaned version as a new CSV
    """
    encodings_to_try = ["utf-8", "cp1252", "ISO-8859-1"]
    df = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(input_csv_path, encoding=enc, on_bad_lines="skip", engine="python")
            print(f"✅ Loaded CSV using encoding: {enc}")
            break
        except UnicodeDecodeError:
            print(f"⚠️ Failed with encoding: {enc}, trying next...")
    
    if df is None:
        raise ValueError("❌ Could not load the CSV with the tried encodings.")

    def clean_text(text):
        if pd.isna(text):
            return text
        text = str(text)

        # --- HARD REMOVE ALL DOUBLE QUOTES so """ cannot appear ---
        # (removes any sequence of one or more " characters)
        text = re.sub(r'"+', '', text)

        # keep your original cleaning
        text = re.sub(r"[^\w\s.,!?;:'\"()-]", " ", text)  # Remove unwanted characters (quotes already removed)
        text = re.sub(r"\s+", " ", text)                  # Normalize spaces
        return text.strip()

    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].apply(clean_text)

    if not output_csv_path:
        base, ext = os.path.splitext(input_csv_path)
        output_csv_path = f"{base}_cleaned{ext}"

    df.to_csv(output_csv_path, index=False)
    print(f"💾 Cleaned CSV saved to: {output_csv_path}")

    return output_csv_path


# --------------------------
# Example usage in same notebook
# --------------------------
input_path = "/Users/Rex/vscode/ai_lora_tuning/alldata_1_for_kaggle.csv"
cleaned_path = clean_biomed_csv(input_path)

# Load the cleaned CSV to check
df_cleaned = pd.read_csv(cleaned_path)
df_cleaned.head()


⚠️ Failed with encoding: utf-8, trying next...
⚠️ Failed with encoding: cp1252, trying next...
✅ Loaded CSV using encoding: ISO-8859-1
💾 Cleaned CSV saved to: /Users/Rex/vscode/ai_lora_tuning/alldata_1_for_kaggle_cleaned.csv


,Unnamed: 0,0,a
0,0,Thyroid_Cancer,Thyroid surgery in children in a single instit...
1,1,Thyroid_Cancer,The adopted strategy was the same as that used...
2,2,Thyroid_Cancer,coronary arterybypass grafting thrombosis ï br...
3,3,Thyroid_Cancer,Solitary plasmacytoma SP of the skull is an un...
4,4,Thyroid_Cancer,This study aimed to investigate serum matrix m...


In [25]:
df_cleaned.columns

Index(['Unnamed: 0', '0', 'a'], dtype='object')

In [26]:
df_cleaned = df_cleaned.drop(columns=['Unnamed: 0'])
df_cleaned = df_cleaned.rename(columns={'0': 'label', 'a': 'text'})


In [27]:
df_cleaned = df_cleaned[df_cleaned['label'] != "Thyroid_Cancer"]

In [28]:
df_cleaned.head(50)

,label,text
281,Colon_Cancer,the bacteroides fragilis b fragilis produce bi...
282,Colon_Cancer,advances in technology hardware and computing ...
283,Colon_Cancer,bousmalis leveraged cgan with the contentsimil...
284,Colon_Cancer,surfacecontrolled waterdispersible form of cur...
285,Colon_Cancer,during the covid19 pandemic emergency departme...
286,Colon_Cancer,purpose to explore a new therapeutic option fo...
287,Colon_Cancer,this study hypothesizes that bromelain bl acts...
288,Colon_Cancer,malignant mesothelioma is an aggressive cancer...
289,Colon_Cancer,in a novel coronavirus sarscov2 was found to c...
290,Colon_Cancer,netosis is a type of regulated cell death depe...


In [29]:
unique = df_cleaned['label'].value_counts()
print(unique)

label
Colon_Cancer    2580
Lung_Cancer     2180
Name: count, dtype: int64


In [30]:
df_cleaned.columns

Index(['label', 'text'], dtype='object')

In [31]:
# Save DataFrame to CSV
df_cleaned.to_csv("df_cleaned.csv", index=False)
